In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from pathlib import Path
import pickle
# --- 1. Path setup ---
SAVE_DIR = FULL_DATA_ROOT / "100_010"
OUTPUT_BASE = SAVE_DIR / "cfm_output"
# -----------------------------
# 1. Read CFM k^2 data
# -----------------------------
data_file = str(OUTPUT_BASE) + "_k2.dat"

data = np.loadtxt(data_file, comments="#")
k2 = data[:, 0]
ave = 0.5 * (data[:, 1] + data[:, 2])

order = np.argsort(k2)
k2 = k2[order]
ave = ave[order]

# -----------------------------
# 2. Prepare plot data and fit data separately
# -----------------------------
k2_min = 5.0e-3
min_points = 5
L_min_interface = 4
through_origin = True

k_range = [0.07, 0.165]  # this is k range, not k^2 range
k2_fit_min = k_range[0] ** 2
k2_fit_max = k_range[1] ** 2

# Plot all available points within xlim. Do not discard low-k points here.
xlim = [0, 0.07]
ylim = [0, 0.9e-19]

plot_mask = (k2 >= xlim[0]) & (k2 <= xlim[1])
k2_plot = k2[plot_mask]
ave_plot = ave[plot_mask]

# Fit only selected range.
# k2_min is still applied to the fitting data, not the displayed data.
fit_mask = (
    (k2 >= k2_min) &
    (k2 >= k2_fit_min) &
    (k2 <= k2_fit_max)
)

k2_fit_data = k2[fit_mask]
ave_fit_data = ave[fit_mask]

if len(k2_fit_data) < min_points:
    raise ValueError("Not enough points in the selected fitting range.")

# -----------------------------
# 3. Linear fit
# -----------------------------
if through_origin:
    slope = np.linalg.lstsq(k2_fit_data[:, None], ave_fit_data, rcond=None)[0][0]
    intercept = 0.0
else:
    slope, intercept = np.polyfit(k2_fit_data, ave_fit_data, 1)

y_pred = slope * k2_fit_data + intercept
ss_res = np.sum((ave_fit_data - y_pred) ** 2)
ss_tot = np.sum((ave_fit_data - np.mean(ave_fit_data)) ** 2)
r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0

# -----------------------------
# 4. Extra axis data
# -----------------------------
k2_extra = np.array([
    0.0029, 0.00418, 0.00569, 0.00744, 0.00941,
    0.01162, 0.01406, 0.01673, 0.01963, 0.02277,
    0.02614, 0.02974, 0.03358
])

tau_extra = np.array([
    0.0718, 0.0335, 0.0224, 0.019, 0.0131,
    0.0108, 0.0075, 0.0068, 0.0056, 0.0047,
    0.004, 0.0033, 0.003
])

n_eff_extra = np.array([
    69.63788, 149.25373, 223.21429, 263.15789, 381.67939,
    462.96296, 666.66667, 735.29412, 892.85714, 1063.82979,
    1250, 1515.15152, 1666.66667
])

# Use sparse ticks to avoid over-crowding
idx_tau = np.arange(0, len(k2_extra), 2)
idx_neff = np.arange(0, len(k2_extra), 3)

# -----------------------------
# 5. Plot
# -----------------------------
import matplotlib as mpl

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["mathtext.fontset"] = "stix"

# relaxation time: convert to ps
tau_ps_extra = tau_extra * 1000.0

fig, ax = plt.subplots(figsize=(7.5, 8.5))

# All visible data points
ax.plot(k2_plot, ave_plot, "o", color="gray", markersize=7, label="Data")

# Points used for fitting
ax.plot(k2_fit_data, ave_fit_data, "o", color="black", markersize=9, label="Used for fit")

# Fit line
kfit = np.linspace(xlim[0], xlim[1], 300)
yfit = slope * kfit + intercept

fit_label = (
    rf"Fit: $y=mx$, $m={slope:.2e}$, $R^2={r2:.3f}$"
    if through_origin
    else rf"Fit: $y=mx+b$, $m={slope:.2e}$, $b={intercept:.2e}$, $R^2={r2:.3f}$"
)

ax.plot(kfit, yfit, "r--", lw=2.4, label=fit_label)

ax.set_xlabel(r"$k^2$ ($\AA^{-2}$)", fontsize=20)
ax.set_ylabel(
    r"$k_B T/(L_x L_y \langle |A(k)|^2 \rangle)$ (mJ $\AA^{-4}$)",
    fontsize=20
)

ax.set_xlim(xlim)
ax.set_ylim(ylim)

ax.tick_params(axis="both", labelsize=16)
ax.grid(True, ls="--", alpha=0.4)
ax.legend(fontsize=16, frameon=False, loc="upper left")

# -----------------------------
# Top axis 1: relaxation time
# -----------------------------
ax_tau = ax.twiny()
ax_tau.set_xlim(ax.get_xlim())
ax_tau.set_xticks(k2_extra[idx_tau])
ax_tau.set_xticklabels([f"{v:.1f}" for v in tau_ps_extra[idx_tau]], fontsize=14)
ax_tau.set_xlabel("Relaxation time (ps)", fontsize=17, labelpad=10)
ax_tau.tick_params(axis="x", direction="out", pad=4)

# -----------------------------
# Top axis 2: independent sample number
# -----------------------------
ax_neff = ax.twiny()
ax_neff.set_xlim(ax.get_xlim())
ax_neff.spines["top"].set_position(("axes", 1.20))
ax_neff.set_xticks(k2_extra[idx_neff])
ax_neff.set_xticklabels([f"{v:.0f}" for v in n_eff_extra[idx_neff]], fontsize=14)
ax_neff.set_xlabel("Independent samples in 5 ns", fontsize=17, labelpad=10)
ax_neff.tick_params(axis="x", direction="out", pad=4)

plt.tight_layout()
plt.savefig("CFM_k2_with_relaxation_and_samples.svg", format="svg", bbox_inches="tight")
plt.show()
